# VLM-Guided Statistical Model Fitting — Experiments

Supports two domains via the `--domain` flag:
- **Distribution fitting** (`domain="distribution-fitting"`): fit 1-D distributions (gaussian, cauchy, mixtures, etc.)
- **Time series** (`domain="time-series"`): fit Gaussian Process models to temporal data

Each step uses the **agentic tool loop** with two phases:

| Phase | What happens | VLM calls |
|---|---|---|
| **Phase 1: Diagnostic** | VLM is offered tools via native function-calling. It can call 0 to N tools. | 1 `call_for_tool()` per tool call |
| **Phase 2: Proposal** | VLM proposes revised models (or declares COMPLETE). | 1 `call()` for proposals |
| **Code gen** | Each proposal is translated to PyMC code by the VLM. | 1 `call()` per proposal |
| **Summary** | VLM summarizes the step's results for the next iteration. | 1 `call()` for summary |

### Toolkit modes

| Mode | `toolkit_mode` | Dist-fitting tools | Time-series tools | `code_model_config` needed? |
|---|---|---|---|---|
| No tools | `"none"` | Empty | Empty | No |
| Static tools | `"static"` | 5 built-in (moments, GMM, QQ, tails, probability) | 4 built-in (dominant period, fit vs actuals, residuals, ACF) | No |
| Generate only | `"generate_only"` | `generate_new_tool` + registry | **Not supported** | **Yes** |
| Dynamic | `"dynamic"` | Static + `generate_new_tool` + registry | **Not supported** | **Yes** |

### Experiment matrix

| # | Domain | Data | `toolkit_mode` | `force_tool_call` | Key question |
|---|---|---|---|---|---|
| 1 | distribution-fitting | cauchy | `none` | No | Baseline: VLM alone |
| 2 | distribution-fitting | cauchy | `static` | No | Do static diagnostics help? |
| 3 | distribution-fitting | cauchy | `generate_only` | Yes | Can the VLM invent diagnostics? |
| 4 | distribution-fitting | cauchy | `dynamic` | No | Best of both |
| 5 | distribution-fitting | gaussian+cauchy | `none` | No | Mixture baseline |
| 6 | distribution-fitting | gaussian+cauchy | `static` | No | Do tools help with mixtures? |
| 7 | time-series | no anomaly | `none` | No | TS baseline: VLM alone |
| 8 | time-series | no anomaly | `static` | No | Do TS diagnostics help? |

## Setup

In [ ]:
import logging
import os
import pickle
import sys
import warnings
from pathlib import Path
from typing import Any, Dict, List

warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")
warnings.filterwarnings("ignore", category=FutureWarning)

import matplotlib

matplotlib.use("Agg")
from dotenv import load_dotenv
from IPython.display import Image, display

PKG_DIR: Path = Path(".").resolve()
if str(PKG_DIR.parent) not in sys.path:
    sys.path.insert(0, str(PKG_DIR.parent))
load_dotenv(PKG_DIR / ".env")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s.%(msecs)03d [%(name)-20s] %(levelname)-7s %(message)s",
    datefmt="%Y-%m-%d::%H:%M:%S",
)
for _name in (
    "matplotlib",
    "PIL",
    "urllib3",
    "httpcore",
    "httpx",
    "openai",
    "litellm",
    "LiteLLM",
    "numba",
    "asyncio",
    "arviz",
    "pymc",
    "pytensor",
    "filelock",
):
    logging.getLogger(_name).setLevel(logging.WARNING)

from pymc_model_selection.experiments import run

# Load distribution-fitting datasets
with open(PKG_DIR / "data_single.pkl", "rb") as f:
    SINGLE_DATASETS: List[Dict[str, Any]] = pickle.load(f)
with open(PKG_DIR / "data_mixed.pkl", "rb") as f:
    MIXED_DATASETS: List[Dict[str, Any]] = pickle.load(f)
print(f"Distribution fitting: {len(SINGLE_DATASETS)} single + {len(MIXED_DATASETS)} mixed datasets")

# Load time-series datasets
with open(PKG_DIR / "dataset_ts_no_anomaly_medium.pkl", "rb") as f:
    TS_DATASETS: List[Dict[str, Any]] = pickle.load(f)
print(f"Time series: {len(TS_DATASETS)} datasets")

## Model config

Uncomment exactly **one** `MODEL_CONFIG` block.

In [ ]:
import sys
from pathlib import Path

PKG_DIR = Path(".").resolve()
if str(PKG_DIR) not in sys.path:
    sys.path.insert(0, str(PKG_DIR))
from dotenv import load_dotenv

load_dotenv(PKG_DIR / ".env")
from vlm_backends import VLMBackend

print("=== azure/gpt-5-mini smoke test (30s timeout, 3 retries) ===")
try:
    b = VLMBackend.of(
        "api",
        litellm_model="azure/gpt-5-mini",
        max_tokens=256,
        temperature=0.5,
        verbosity=1,
        num_retries=3,
        call_timeout=30.0,
    )
    r = b.call(
        prompt='Return ONLY a valid JSON object: {"test": "hello"}',
        verbosity=1,
    )
    print(f"SUCCESS: response={r!r}")
    b.stop()
except Exception as e:
    print(f"ERROR: {type(e).__name__}: {e}")

In [2]:
# ── Experiment Configuration ──────────────────────────────────────────
MODEL: ModelConfig = ModelConfig(litellm_model="azure/gpt-5-mini")

# # Alternative: Together AI
# MODEL: ModelConfig = ModelConfig(
#     litellm_model="together_ai/Qwen/Qwen3.5-9B",
#     max_tokens=4096,
#     temperature=0.3,
#     litellm_params={"reasoning_effort": "low", "allowed_openai_params": ["tools", "tool_choice"]},
# )

MAX_STEPS: int = 3

print(f"VLM model: {MODEL.litellm_model}")
print(f"Max steps: {MAX_STEPS}")

VLM model:     azure/gpt-5-mini
Code-gen model: azure/gpt-5-mini
Max steps:      3


## Helper: display step results

In [3]:
def show_result(result: Dict[str, Any]) -> None:
    """Print summary and display step images for a single run() result."""
    status: str = result["status"]
    true_dist: str = result["true_distribution"]
    final_family: str = result.get("final_family", "N/A")
    final_aic: Any = result.get("final_aic")
    num_steps: int = len(result["steps"])

    print(f"Status: {status}")
    print(f"True: {true_dist}  →  Predicted: {final_family}")
    if isinstance(final_aic, float):
        print(f"Final AIC: {final_aic:.1f}")
    else:
        print(f"Final AIC: {final_aic}")
    print(f"Steps: {num_steps}")
    print()

    for step in result["steps"]:
        step_num: int = step["step"]
        pred: str = step.get("predicted_dist", "?")
        metrics: Dict[str, Any] = step.get("metrics", {})
        aic_str: str = f"{metrics['aic']:.1f}" if "aic" in metrics else "N/A"
        tools: str = step.get("selected_tools", "None")
        print(f"── Step {step_num}: pred={pred}  AIC={aic_str}  tools={tools}")
        if "error" in step:
            print(f"   ERROR: {step['error']}")
        img_path: str = step.get("image_path", "")
        if img_path and os.path.exists(img_path):
            display(Image(filename=img_path, width=600))

---
# Part A: Distribution fitting — single distribution (cauchy)

`domain="distribution-fitting"` with all 4 toolkit modes on the same cauchy distribution.

## Experiment 1: `toolkit_mode="none"` (baseline)

No diagnostic tools. VLM proposes models based on the fit plot only (Phase 1 skipped).

In [ ]:
config_e1: ExperimentConfig = ExperimentConfig(
    model=MODEL,
    toolkit=ToolkitConfig(
        mode="none",
    ),
    output=OutputConfig(expt="notebook_e1"),
    data_pkl="data_single.pkl",
    dataset_idx="0",
    max_steps=MAX_STEPS,
)
result_e1: Dict[str, Any] = run(
    config=config_e1,
    dataset=SINGLE_DATASETS[0],
    out_dir="outputs/notebook_e1",
    verbosity=2,
)
show_result(result_e1)

[SlowBurnAPI] Ready: azure/gpt-5-mini
Backend: SlowBurnAPIBackend / azure/gpt-5-mini
Generated 673 points from 'cauchy'


cauchy → azure/gpt-5-mini:   0%|          | 0/4 [00:00<?, ?step/s]

/Users/adivekar/workplace/pymc_model_selection/plotting_utils.py:400: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



[SlowBurnAPI] ---- Parsed VLM Response ----
  Description: The histogram shows a very sharp, high central peak clustered tightly around zero with a pronounced, long left (negative) tail reaching far below -1000 and only minor mass on the positive side. This suggests a dominant narrow component near zero plus a heavy-tailed component producing extreme negative outliers (i.e., a skewed heavy-tailed mixture). Several plausible explanations are a narrow Gaussian core with a heavy-tailed (student_t or cauchy/laplace) outlier component or a two-Gaussian mixture with one very wide component.
  Toolkit: None
  Models proposed: 0
  ------------------------------------

[SlowBurnAPI] Response (756 chars):
{"code": "with pm.Model() as model:\n    w_gaussian_student_t = pm.Dirichlet('w_gaussian_student_t', a=np.array([5.0, 1.0]))\n    gaussian_mu_0 = pm.Normal('gaussian_mu_0', mu=0.0, sigma=1.0)\n    gaussian_sigma_0 = pm.HalfNormal('gaussian_sigma_0', sigma=1.0)\n    student_t_mu_1 = pm.Normal('

Output()

...

/Users/adivekar/workplace/pymc_model_selection/exploration_toolkit.py:489: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/adivekar/miniconda3/envs/pymc/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:4232: RuntimeWarning: Degrees of freedom <= 0 for slice
  return _methods._var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Users/adivekar/miniconda3/envs/pymc/lib/python3.13/site-packages/numpy/_core/_methods.py:211: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


  Component 0 (weight=0.997): mean=0.8258  std=25.68  skew=7.831  kurt=101.5
    Symmetry : right-skewed (skew=7.83) → Gamma, Lognormal, Weibull, Exponential, Chi-squared
    Tails    : very leptokurtic (fat tails) → Student-t (small ν), Cauchy, Lognormal
  Component 1 (weight=0.001): mean=-2089  std=nan  skew=nan  kurt=nan
    Symmetry : left-skewed (skew=nan) → reflected Gamma/Weibull, Beta (α>β)
    Tails    : very leptokurtic (fat tails) → Student-t (small ν), Cauchy, Lognormal
  Component 2 (weight=0.001): mean=-322.7  std=nan  skew=nan  kurt=nan
    Symmetry : left-skewed (skew=nan) → reflected Gamma/Weibull, Beta (α>β)
    Tails    : very leptokurtic (fat tails) → Student-t (small ν), Cauchy, Lognormal
  Moment Summary
  Mean              : -2.761
  Variance          : 7302
  Skewness          : -21.54
  Excess Kurtosis   : 528.1
-------------------------------------------------------
  Symmetry hint     : left-skewed (skew=-21.54) → reflected Gamma/Weibull, Beta (α>β)
  Tail-we

Output()

MAP ESTIMATE: {'w_student_t_student_t_student_t_simplex__': array([ 31.78949339, -11.71702108]), 'student_t_mu_0': array(-0.59679393), 'student_t_sigma_0_log__': array(0.51755613), 'student_t_nu_0_log__': array(-0.09093513), 'student_t_mu_1': array(-2088.99873388), 'student_t_sigma_1_log__': array(-7.35965525), 'student_t_nu_1_log__': array(-10.58654688), 'student_t_mu_2': array(-322.69744499), 'student_t_sigma_2_log__': array(-2.30370009), 'student_t_nu_2_log__': array(-11.74759217), 'w_student_t_student_t_student_t': array([1.00000000e+00, 1.27456162e-19, 2.99663935e-23]), 'student_t_sigma_0': array(1.67792202), 'student_t_nu_0': array(0.91307694), 'student_t_sigma_1': array(0.00063642), 'student_t_nu_1': array(2.52534726e-05), 'student_t_sigma_2': array(0.09988856), 'student_t_nu_2': array(7.90834384e-06)}
Model 0 fit metrics: {'aic': np.float64(4584.424015370828), 'bic': np.float64(4634.05321399692), 'n_params': 11}

BEST MODEL: 0
Distribution family: ['student_t', 'student_t', 'st

Output()

## Experiment 2: `toolkit_mode="static"` (5 built-in tools)

Phase 1 offers `calculate_moments`, `segment_distributions_and_calculate_moments`, `qq_plot`, `plot_tails_transform`, `probability_plot` via native function-calling. VLM decides whether to call any.

In [ ]:
config_e2: ExperimentConfig = ExperimentConfig(
    model=MODEL,
    toolkit=ToolkitConfig(
        mode="static",
    ),
    output=OutputConfig(expt="notebook_e2"),
    data_pkl="data_single.pkl",
    dataset_idx="0",
    max_steps=MAX_STEPS,
)
result_e2: Dict[str, Any] = run(
    config=config_e2,
    dataset=SINGLE_DATASETS[0],
    out_dir="outputs/notebook_e2",
    verbosity=2,
)
show_result(result_e2)

## Experiment 3: `toolkit_mode="generate_only"` + `force_tool_call=True`

Phase 1 offers only `generate_new_tool` (+ any previously generated registry tools). No static tools.
`force_tool_call=True` forces the VLM to call `generate_new_tool` on the first diagnostic turn of every step. A separate `code_gen_backend` (VLMBackend instance) writes the tool code.

In [ ]:
config_e3: ExperimentConfig = ExperimentConfig(
    model=MODEL,
    toolkit=ToolkitConfig(
        mode="generate_only",
        code_gen_model="azure/gpt-5-mini",
    ),
    output=OutputConfig(expt="notebook_e3"),
    data_pkl="data_single.pkl",
    dataset_idx="0",
    max_steps=MAX_STEPS,
)
result_e3: Dict[str, Any] = run(
    config=config_e3,
    dataset=SINGLE_DATASETS[0],
    out_dir="outputs/notebook_e3",
    verbosity=2,
)
show_result(result_e3)

## Experiment 4: `toolkit_mode="dynamic"` (static + generation)

Phase 1 offers all 5 static tools + `generate_new_tool` + any registry tools.
VLM chooses whether to use a built-in tool, generate a new one, or skip diagnostics entirely.

In [ ]:
config_e4: ExperimentConfig = ExperimentConfig(
    model=MODEL,
    toolkit=ToolkitConfig(
        mode="dynamic",
        code_gen_model="azure/gpt-5-mini",
    ),
    output=OutputConfig(expt="notebook_e4"),
    data_pkl="data_single.pkl",
    dataset_idx="0",
    max_steps=MAX_STEPS,
)
result_e4: Dict[str, Any] = run(
    config=config_e4,
    dataset=SINGLE_DATASETS[0],
    out_dir="outputs/notebook_e4",
    verbosity=2,
)
show_result(result_e4)

---
# Part B: Distribution fitting — mixture (gaussian + cauchy)

Same domain, but on a two-component mixture. Only `none` and `static` (no generate_only/dynamic for mixture since static tools already cover the GMM case).

## Experiment 5: Mixture, `toolkit_mode="none"`

In [ ]:
print(f"Mixed datasets: {len(MIXED_DATASETS)}")
print(f"First: dist_choice={MIXED_DATASETS[0]['dist_choice']}, n={len(MIXED_DATASETS[0]['data'])}")

result_e5: Dict[str, Any] = run(
    domain="distribution-fitting",
    dataset=MIXED_DATASETS[0],
    model_config=MODEL_CONFIG,
    max_steps=MAX_STEPS,
    toolkit_mode="none",
    verbosity=2,
)
show_result(result_e5)

## Experiment 6: Mixture, `toolkit_mode="static"`

In [ ]:
config_e6: ExperimentConfig = ExperimentConfig(
    model=MODEL,
    toolkit=ToolkitConfig(
        mode="static",
    ),
    output=OutputConfig(expt="notebook_e6"),
    data_pkl="data_mixed.pkl",
    dataset_idx="0",
    max_steps=MAX_STEPS,
)
result_e6: Dict[str, Any] = run(
    config=config_e6,
    dataset=MIXED_DATASETS[0],
    out_dir="outputs/notebook_e6",
    verbosity=2,
)
show_result(result_e6)

---
# Part C: Time series — GP fitting (no anomaly)

`domain="time-series"` supports `none` and `static` toolkit modes.
`generate_only` and `dynamic` are not supported for time series (the TS domain rejects them with a clear error).

**Static TS tools:** `plot_dominant_period`, `plot_fit_vs_actuals`, `plot_residuals`, `plot_acf_pacf`.

## Experiment 7: Time series, `toolkit_mode="none"` (baseline)

No diagnostic tools. VLM can only propose GP kernel combinations based on the fit plot.

In [ ]:
print(
    f"Time-series dataset 0: name={TS_DATASETS[0]['name']}, "
    f"category={TS_DATASETS[0]['category']}, "
    f"anomaly={TS_DATASETS[0]['anomaly_info']}, "
    f"n={len(TS_DATASETS[0]['data'])}"
)

result_e7: Dict[str, Any] = run(
    domain="time-series",
    dataset=TS_DATASETS[0],
    model_config=MODEL_CONFIG,
    max_steps=MAX_STEPS,
    toolkit_mode="none",
    verbosity=2,
)
show_result(result_e7)

## Experiment 8: Time series, `toolkit_mode="static"`

VLM sees 4 TS-specific diagnostics: dominant period analysis, fit vs actuals overlay, residual plot, ACF/PACF.

In [ ]:
config_e8: ExperimentConfig = ExperimentConfig(
    model=MODEL,
    toolkit=ToolkitConfig(
        mode="static",
    ),
    output=OutputConfig(expt="notebook_e8"),
    data_pkl="dataset_ts_no_anomaly_medium.pkl",
    dataset_idx="0",
    max_steps=MAX_STEPS,
)
result_e8: Dict[str, Any] = run(
    config=config_e8,
    dataset=TS_DATASETS[0],
    out_dir="outputs/notebook_e8",
    verbosity=2,
)
show_result(result_e8)

---
# Part D: Comparison summary

Collect all results for side-by-side comparison.

In [ ]:
import pandas as pd

ALL_RESULTS: List[Dict[str, Any]] = [
    {
        "experiment": "E1 dist-fit/none",
        "domain": "distribution-fitting",
        "toolkit": "none",
        **{k: result_e1.get(k) for k in ("status", "true_distribution", "final_family", "final_aic")},
    },
    {
        "experiment": "E2 dist-fit/static",
        "domain": "distribution-fitting",
        "toolkit": "static",
        **{k: result_e2.get(k) for k in ("status", "true_distribution", "final_family", "final_aic")},
    },
    {
        "experiment": "E3 dist-fit/generate_only",
        "domain": "distribution-fitting",
        "toolkit": "generate_only",
        **{k: result_e3.get(k) for k in ("status", "true_distribution", "final_family", "final_aic")},
    },
    {
        "experiment": "E4 dist-fit/dynamic",
        "domain": "distribution-fitting",
        "toolkit": "dynamic",
        **{k: result_e4.get(k) for k in ("status", "true_distribution", "final_family", "final_aic")},
    },
    {
        "experiment": "E5 dist-fit-mix/none",
        "domain": "distribution-fitting",
        "toolkit": "none",
        **{k: result_e5.get(k) for k in ("status", "true_distribution", "final_family", "final_aic")},
    },
    {
        "experiment": "E6 dist-fit-mix/static",
        "domain": "distribution-fitting",
        "toolkit": "static",
        **{k: result_e6.get(k) for k in ("status", "true_distribution", "final_family", "final_aic")},
    },
    {
        "experiment": "E7 time-series/none",
        "domain": "time-series",
        "toolkit": "none",
        **{k: result_e7.get(k) for k in ("status", "true_distribution", "final_family", "final_aic")},
    },
    {
        "experiment": "E8 time-series/static",
        "domain": "time-series",
        "toolkit": "static",
        **{k: result_e8.get(k) for k in ("status", "true_distribution", "final_family", "final_aic")},
    },
]

df_comparison: pd.DataFrame = pd.DataFrame(ALL_RESULTS)
df_comparison["num_steps"] = [
    len(r["steps"])
    for r in [result_e1, result_e2, result_e3, result_e4, result_e5, result_e6, result_e7, result_e8]
]
df_comparison